In [ ]:
# BEFORE RUNNING THIS NOTEBOOK, create the merged StationXML once:
#   python QQ_merge_stationxml.py --data_root <your project data folder>

# %% [markdown]
# # Classification of volcanic seismicity with the Frequency Index (FI)
# Clean, modular version
# - no P-wave refinement
# - uses spectral energy
# - all settings in the configuration cell below
# - automatic classification from the FI histogram
# - final merge by event_id

# %% [markdown]
# ## 1. Imports and configuration

# %%
import json
from pathlib import Path
from functools import lru_cache
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

from obspy import read, read_inventory, UTCDateTime
from obspy.geodetics import gps2dist_azimuth
from obspy.signal.util import next_pow_2

from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings("ignore")


In [ ]:

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
# Lines to edit: where your project's results are.
# Relative paths are relative to this notebook's folder.
DATA_DIR = Path("../../examples/pavlof_2021/data").resolve()   # the "data" folder next to your QQ_driver.py
NAMEBASE = "GAMMA"                                             # same as location_namebase in QQ_driver.py
MERGED_DIR = DATA_DIR / "merged"

PATHS = {
    "data_base": DATA_DIR,
    "catalog_csv": MERGED_DIR / "output" / f"{NAMEBASE}_hypodd_catalog_qcfiltered.csv",
    "picks_csv": MERGED_DIR / "input" / f"{NAMEBASE}_picks_cleaned.csv",
    "stations_json": MERGED_DIR / "input" / f"{NAMEBASE}_station_list.json",
    "inventory_xml": MERGED_DIR / "input" / "stations_merged.xml",
    "output_csv": MERGED_DIR / "output" / f"{NAMEBASE}_hypodd_catalog_qcfiltered_FI.csv",
    "output_dir": MERGED_DIR / "output" / "fi_outputs",
}

# Volcano (or target) used for distance filters: edit for your region

CONFIG = {
    "volcano": {
        "name": "pavlof_2021",
        "lat": 55.4173,
        "lon": -161.8937,
    },
    "filters": {
        "event_radius_km": 100.0,
        "station_max_dist_km": 67.0,
        "min_stations": 2,
        "snr_threshold": 2.0,
    },
    "windows": {
        "p_rel_start_s": -1.0,
        "p_win_len_s": 7.0,
        "noise_win_s": 5.0,
    },
    "frequency_bands": {
        "low": (1.0, 5.0),
        "high": (5.0, 15.0),
    },
    "classification": {
        "method": "auto_hist",
        "auto_min_events": 20,
        "hist_bins_auto": 12,
        "edge_exclusion_fraction": 0.10,
        "smooth_sigma": 1.0,
        "labels": {
            "low_freq": "LP",
            "high_freq": "VT",
            "unclassified": "NS",
        },
    },
    "plot": {
        "hist_bins": 20,
    },
}

VOLCANO = CONFIG["volcano"]
FILTERS = CONFIG["filters"]
WINDOWS = CONFIG["windows"]
FREQ_BANDS = CONFIG["frequency_bands"]
CLASSIFICATION = CONFIG["classification"]
PLOT = CONFIG["plot"]

PATHS["output_dir"].mkdir(parents=True, exist_ok=True)


In [ ]:

# %% [markdown]
# ## 2. Helper functions

# %%
def normalize_event_id(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r"\.0+$", "", regex=True)
    s = s.str.replace(r"\s+", "", regex=True)
    return s

def horizontal_distance_km(lat1, lon1, lat2, lon2):
    dist_m, _, _ = gps2dist_azimuth(lat1, lon1, lat2, lon2)
    return dist_m / 1000.0

def get_hypocentral_distances(stations_df, ev_lat, ev_lon, ev_depth_km):
    stations_df = stations_df.copy()
    if "elev_km" not in stations_df.columns:
        stations_df["elev_km"] = 0.0

    hypo_distances = []
    for _, sta in stations_df.iterrows():
        dh_km = horizontal_distance_km(ev_lat, ev_lon, float(sta["lat"]), float(sta["lon"]))
        dz_km = ev_depth_km + float(sta["elev_km"])
        d3_km = np.sqrt(dh_km**2 + dz_km**2)
        hypo_distances.append(d3_km)

    stations_df["hypo_km"] = hypo_distances
    return dict(zip(stations_df["sta_id"], stations_df["hypo_km"]))

def filter_catalog_by_radius(df, volcano, radius_km, lat_col="latitude", lon_col="longitude"):
    out = df.copy()
    out["r_km"] = [
        horizontal_distance_km(lat, lon, volcano["lat"], volcano["lon"])
        for lat, lon in zip(out[lat_col], out[lon_col])
    ]
    out["inside_radius"] = out["r_km"] <= radius_km
    out_filtered = out[out["inside_radius"]].copy().reset_index(drop=True)
    return out, out_filtered

def infer_catalog_columns(df):
    colmap = {}
    candidates = {
        "time": ["datetime", "time", "origin_time"],
        "latitude": ["latitude", "lat"],
        "longitude": ["longitude", "lon"],
        "depth_km": ["depth_km", "depth"],
        "event_id": ["event_id", "id"],
        "magnitude": ["mag", "magnitude"],
    }

    for key, options in candidates.items():
        found = None
        for col in options:
            if col in df.columns:
                found = col
                break
        colmap[key] = found

    missing_required = [k for k in ["time", "latitude", "longitude", "event_id"] if colmap[k] is None]
    if missing_required:
        raise ValueError(
            f"Catalog is missing required columns: {missing_required}. "
            f"Available columns: {list(df.columns)}"
        )
    return colmap

def load_catalog(catalog_csv):
    catalog = pd.read_csv(catalog_csv)
    colmap = infer_catalog_columns(catalog)

    catalog[colmap["time"]] = pd.to_datetime(
        catalog[colmap["time"]], utc=True, errors="coerce"
    )
    catalog = catalog.dropna(subset=[colmap["time"]]).reset_index(drop=True)
    catalog[colmap["event_id"]] = normalize_event_id(catalog[colmap["event_id"]])

    return catalog, colmap

def load_picks(picks_csv):
    picks = pd.read_csv(picks_csv)

    if "timestamp" not in picks.columns:
        raise ValueError("The picks file must contain a 'timestamp' column.")
    picks["timestamp"] = pd.to_datetime(picks["timestamp"], utc=True, errors="coerce")
    picks = picks.dropna(subset=["timestamp"]).reset_index(drop=True)

    if "id" not in picks.columns:
        raise ValueError("The picks file must contain an 'id' column like AV.STA..BH")

    id_parts = picks["id"].astype(str).str.split(".", expand=True)
    if id_parts.shape[1] < 2:
        raise ValueError("Could not parse 'id' column to extract net and sta.")

    picks["net"] = id_parts[0]
    picks["sta"] = id_parts[1]
    picks["sta_id"] = picks["net"].astype(str) + "." + picks["sta"].astype(str)

    if "type" not in picks.columns:
        raise ValueError("The picks file must contain a 'type' column.")
    picks["type"] = picks["type"].astype(str).str.upper()

    if "event_id_mapped" in picks.columns:
        picks["event_id_link"] = normalize_event_id(picks["event_id_mapped"])
    elif "event_id" in picks.columns:
        picks["event_id_link"] = normalize_event_id(picks["event_id"])
    else:
        raise ValueError("The picks file must contain 'event_id_mapped' or 'event_id'.")

    return picks

def load_stations(stations_json):
    with open(stations_json, "r") as f:
        stations_raw = json.load(f)

    stations_list = []
    for code, values in stations_raw.items():
        parts = code.split(".")
        net = parts[0] if len(parts) > 0 else ""
        sta = parts[1] if len(parts) > 1 else ""
        sta_id = f"{net}.{sta}" if net and sta else code

        elev_m = None
        if "elevation(m)" in values:
            elev_m = values["elevation(m)"]
        elif "elev_m" in values:
            elev_m = values["elev_m"]
        elif "elevation_m" in values:
            elev_m = values["elevation_m"]

        elev_km = float(elev_m) / 1000.0 if elev_m is not None else 0.0

        stations_list.append({
            "sta_id": sta_id,
            "net": net,
            "sta": sta,
            "lat": values.get("latitude"),
            "lon": values.get("longitude"),
            "elev_km": elev_km,
        })

    stations = pd.DataFrame(stations_list)

    required_cols = ["sta_id", "lat", "lon", "elev_km"]
    for col in required_cols:
        if col not in stations.columns:
            raise ValueError(f"Missing required station column: {col}")

    return stations

def load_inventory(inventory_xml):
    inventory_xml = Path(inventory_xml)
    if not inventory_xml.exists():
        print(f"Inventory file not found: {inventory_xml}")
        return None

    try:
        return read_inventory(str(inventory_xml))
    except Exception as e:
        print(f"Error loading inventory XML: {e}")
        return None


In [ ]:

# %% [markdown]
# ## 3. Signal processing and FI computation

# %%
def calculate_snr(signal_data, noise_data):
    if len(signal_data) == 0 or len(noise_data) == 0:
        return 0.0

    signal_rms = np.sqrt(np.mean(np.asarray(signal_data, dtype=float) ** 2))
    noise_rms = np.sqrt(np.mean(np.asarray(noise_data, dtype=float) ** 2))

    if noise_rms == 0:
        return np.inf
    return signal_rms / noise_rms

def preprocess_trace(trace):
    tr = trace.copy()
    tr.data = tr.data.astype(float)
    tr.detrend("demean")
    tr.detrend("linear")
    tr.taper(max_percentage=0.05, type="hann")
    return tr

def remove_instrument_response(trace, inventory):
    corrected_trace = trace.copy()
    try:
        fs = corrected_trace.stats.sampling_rate
        nyq = 0.5 * fs

        f1 = 0.5
        f2 = 1.0
        f3 = min(0.9 * nyq, nyq - 1.0)
        f4 = min(0.95 * nyq, nyq - 0.1)

        if not (f1 < f2 < f3 < f4):
            return corrected_trace, False

        corrected_trace.remove_response(
            inventory=inventory,
            output="VEL",
            water_level=60,
            pre_filt=(f1, f2, f3, f4),
        )
        return corrected_trace, True

    except Exception:
        try:
            corrected_trace = trace.copy()
            corrected_trace.remove_response(
                inventory=inventory,
                output="VEL",
                water_level=60,
            )
            return corrected_trace, True
        except Exception:
            return trace, False

def extract_signal_and_noise_windows(trace, anchor_time, windows):
    signal_start = anchor_time + windows["p_rel_start_s"]
    signal_end = signal_start + windows["p_win_len_s"]

    noise_start = signal_start - windows["noise_win_s"]
    noise_end = signal_start

    signal_trace = trace.copy()
    signal_trace.trim(signal_start, signal_end, pad=True, fill_value=0.0)

    noise_trace = trace.copy()
    noise_trace.trim(noise_start, noise_end, pad=True, fill_value=0.0)

    return signal_trace, noise_trace

def compute_amplitude_spectrum(trace):
    data = trace.data.astype(float)
    npts = len(data)

    if npts < 10:
        return None, None
    if not np.all(np.isfinite(data)):
        return None, None

    window = np.hanning(npts)
    data_windowed = data * window

    nfft = next_pow_2(npts)
    spectrum = np.fft.rfft(data_windowed, n=nfft)
    freqs = np.fft.rfftfreq(nfft, d=trace.stats.delta)
    amp = np.abs(spectrum)

    return freqs, amp

def compute_station_fi(trace, p_arrival_time, windows, snr_threshold, freq_bands, inventory=None):
    """
    Compute the Frequency Index (FI) with the classical definition:
    FI = log10(mean(A_high) / mean(A_low))

    where:
    - A_high = mean spectral amplitude in the high band
    - A_low  = mean spectral amplitude in the low band
    """
    try:
        anchor_time = p_arrival_time

        signal_trace, noise_trace = extract_signal_and_noise_windows(
            trace, anchor_time, windows
        )

        snr = calculate_snr(
            signal_trace.data.astype(float),
            noise_trace.data.astype(float)
        )
        if snr < snr_threshold:
            return None

        processed = preprocess_trace(signal_trace)

        if inventory is not None:
            processed, response_removed = remove_instrument_response(processed, inventory)
            if not response_removed:
                return None

        if np.any(~np.isfinite(processed.data)):
            return None

        freqs, amp = compute_amplitude_spectrum(processed)
        if freqs is None or amp is None:
            return None

        f_low_min, f_low_max = freq_bands["low"]
        f_high_min, f_high_max = freq_bands["high"]

        low_mask = (freqs >= f_low_min) & (freqs <= f_low_max)
        high_mask = (freqs >= f_high_min) & (freqs <= f_high_max)

        if not np.any(low_mask) or not np.any(high_mask):
            return None

        a_low = np.mean(amp[low_mask])
        a_high = np.mean(amp[high_mask])

        if not np.isfinite(a_low) or not np.isfinite(a_high):
            return None
        if a_low <= 0:
            return None

        fi = np.log10(a_high / a_low)

        return {
            "fi": float(fi),
            "snr": float(snr),
            "a_low": float(a_low),
            "a_high": float(a_high),
        }

    except Exception:
        return None


In [ ]:

# %% [markdown]
# ## 4. Chunks, picks per event and per-event processing

# %%
def build_chunk_index(data_base):
    chunk_rows = []
    for p in sorted(Path(data_base).iterdir()):
        if not p.is_dir():
            continue
        if p.name == "merged":
            continue
        try:
            start = pd.to_datetime(p.name, format="%Y%m%dT%H%M%S", utc=True)
            chunk_rows.append({"folder": p.name, "start": start})
        except Exception:
            continue

    chunk_df = pd.DataFrame(chunk_rows).sort_values("start").reset_index(drop=True)
    if len(chunk_df) == 0:
        raise ValueError(f"No valid chunk folders found in {data_base}")

    chunk_df["end"] = chunk_df["start"].shift(-1)

    if len(chunk_df) > 1:
        last_duration = chunk_df.loc[len(chunk_df)-2, "end"] - chunk_df.loc[len(chunk_df)-2, "start"]
    else:
        last_duration = pd.Timedelta(hours=8)

    chunk_df.loc[len(chunk_df)-1, "end"] = chunk_df.loc[len(chunk_df)-1, "start"] + last_duration
    return chunk_df

def prepare_chunk_lookup(chunk_index):
    starts = chunk_index["start"].dt.tz_convert("UTC").dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
    ends = chunk_index["end"].dt.tz_convert("UTC").dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
    folders = chunk_index["folder"].to_numpy()
    return starts, ends, folders

def find_chunk_folder_for_event_fast(event_time, chunk_starts, chunk_ends, chunk_folders):
    t = np.datetime64(pd.Timestamp(event_time).tz_convert("UTC").tz_localize(None))
    pos = np.searchsorted(chunk_starts, t, side="right") - 1
    if pos < 0:
        return None
    if t < chunk_ends[pos]:
        return chunk_folders[pos]
    return None

@lru_cache(maxsize=32)
def read_waveform_folder_cached(data_base_str, waveform_folder_name):
    folder = Path(data_base_str) / waveform_folder_name / "waveforms"
    pattern = str(folder / "*.mseed")
    try:
        stream = read(pattern)
        return stream
    except Exception:
        return None

def build_picks_by_event_lookup(picks):
    out = {}
    picks_p = picks[picks["type"] == "P"].copy()

    for ev_id, grp in picks_p.groupby("event_id_link"):
        g = grp.copy()

        if "prob" in g.columns:
            g = (
                g.sort_values("prob", ascending=False)
                .groupby("sta_id", as_index=False)
                .first()
            )
        else:
            g = g.groupby("sta_id", as_index=False).first()

        out[str(ev_id)] = g.reset_index(drop=True)

    return out

def build_event_station_distance_lookup(catalog_filtered, stations, catalog_colmap):
    lookup = {}
    station_info = stations[["sta_id", "lat", "lon", "elev_km"]].copy()

    depth_col = catalog_colmap.get("depth_km")
    event_id_col = catalog_colmap["event_id"]

    for _, ev in catalog_filtered.iterrows():
        ev_id = str(ev[event_id_col])
        ev_lat = float(ev[catalog_colmap["latitude"]])
        ev_lon = float(ev[catalog_colmap["longitude"]])
        ev_depth = float(ev[depth_col]) if depth_col is not None else 0.0

        lookup[ev_id] = get_hypocentral_distances(
            station_info, ev_lat, ev_lon, ev_depth
        )

    return lookup

def process_event(
    event_row,
    picks_by_event,
    event_station_distance_lookup,
    data_base,
    windows,
    filters,
    freq_bands,
    inventory,
    catalog_colmap,
    chunk_starts,
    chunk_ends,
    chunk_folders,
):
    event_time = pd.Timestamp(event_row[catalog_colmap["time"]])
    event_id = str(event_row[catalog_colmap["event_id"]])
    depth_col = catalog_colmap.get("depth_km")

    result = {
        "event_id": event_id,
        "event_time": event_time,
        "event_depth_km": float(event_row[depth_col]) if depth_col is not None else np.nan,
        "r_km": event_row.get("r_km", np.nan),
        "fi": np.nan,
        "fi_std": np.nan,
        "fi_n_stations": 0,
        "fi_snr_mean": np.nan,
        "status": "init",
    }

    if ("original_folder" in event_row) and pd.notna(event_row["original_folder"]):
        waveform_folder_name = str(event_row["original_folder"])
    else:
        waveform_folder_name = find_chunk_folder_for_event_fast(
            event_time, chunk_starts, chunk_ends, chunk_folders
        )

    if waveform_folder_name is None:
        result["status"] = "missing_waveform_folder"
        return result

    stream = read_waveform_folder_cached(str(data_base), waveform_folder_name)
    if stream is None:
        result["status"] = "waveform_read_error"
        return result
    if len(stream) == 0:
        result["status"] = "no_waveforms"
        return result

    event_picks = picks_by_event.get(event_id)
    if event_picks is None or event_picks.empty:
        result["status"] = "no_event_picks"
        return result

    distance_map = event_station_distance_lookup.get(event_id, {})
    event_picks = event_picks.copy()

    event_picks["hypo_km"] = event_picks["sta_id"].map(distance_map)
    event_picks = event_picks[event_picks["hypo_km"].notna()].copy()
    event_picks = event_picks[event_picks["hypo_km"] <= filters["station_max_dist_km"]].copy()

    if event_picks.empty:
        result["status"] = "no_picks_within_station_distance"
        return result

    station_fi = []
    station_snr = []

    for _, pick in event_picks.iterrows():
        sta = str(pick["sta"])
        p_arrival = UTCDateTime(pd.Timestamp(pick["timestamp"]).to_pydatetime())

        traces = stream.select(station=sta, channel="*Z")
        if len(traces) == 0:
            continue

        traces_sorted = sorted(traces, key=lambda tr: tr.stats.sampling_rate, reverse=True)
        trace = traces_sorted[0].copy()

        fi_info = compute_station_fi(
            trace=trace,
            p_arrival_time=p_arrival,
            windows=windows,
            snr_threshold=filters["snr_threshold"],
            freq_bands=freq_bands,
            inventory=inventory,
        )

        if fi_info is None:
            continue

        station_fi.append(fi_info["fi"])
        station_snr.append(fi_info["snr"])

    if len(station_fi) < filters["min_stations"]:
        result["status"] = "not_enough_valid_stations"
        return result

    result["fi"] = float(np.mean(station_fi))
    result["fi_std"] = float(np.std(station_fi)) if len(station_fi) > 1 else 0.0
    result["fi_n_stations"] = int(len(station_fi))
    result["fi_snr_mean"] = float(np.mean(station_snr)) if station_snr else np.nan
    result["status"] = "ok"

    return result


In [ ]:

# %% [markdown]
# ## 5. Histogram-based classification

# %%
def find_fi_threshold_from_histogram(fi_values, classification_cfg):
    fi_values = np.asarray(fi_values, dtype=float)
    fi_values = fi_values[np.isfinite(fi_values)]

    min_events = classification_cfg["auto_min_events"]
    hist_bins = classification_cfg["hist_bins_auto"]
    edge_exclusion_fraction = classification_cfg["edge_exclusion_fraction"]
    smooth_sigma = classification_cfg["smooth_sigma"]

    debug = {
        "counts": None,
        "counts_smooth": None,
        "bin_edges": None,
        "bin_centers": None,
        "minima": np.array([], dtype=int),
        "internal_minima": np.array([], dtype=int),
        "selected_minimum": None,
    }

    if len(fi_values) < min_events:
        return None, "too_few_events", debug

    if np.nanmin(fi_values) == np.nanmax(fi_values):
        return None, "degenerate_distribution", debug

    try:
        counts, bin_edges = np.histogram(fi_values, bins=hist_bins, density=False)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        debug["counts"] = counts
        debug["bin_edges"] = bin_edges
        debug["bin_centers"] = bin_centers

        if smooth_sigma is not None and smooth_sigma > 0:
            counts_smooth = gaussian_filter1d(counts.astype(float), sigma=smooth_sigma)
        else:
            counts_smooth = counts.astype(float)

        debug["counts_smooth"] = counts_smooth

        minima, _ = find_peaks(-counts_smooth)
        debug["minima"] = minima

        if len(minima) == 0:
            return None, "no_local_minimum_found", debug

        n = len(bin_centers)
        left_limit = int(np.floor(edge_exclusion_fraction * n))
        right_limit = int(np.ceil((1 - edge_exclusion_fraction) * n))

        internal_minima = minima[(minima >= left_limit) & (minima < right_limit)]
        debug["internal_minima"] = internal_minima

        if len(internal_minima) == 0:
            return None, "no_internal_minimum_found", debug

        min_idx = internal_minima[np.argmin(counts_smooth[internal_minima])]
        debug["selected_minimum"] = int(min_idx)

        threshold = float(bin_centers[min_idx])
        return threshold, "deepest_internal_histogram_minimum", debug

    except Exception as e:
        return None, f"threshold_error: {e}", debug

def classify_fi_values(valid_df, classification_cfg):
    valid_df = valid_df.copy()

    labels = classification_cfg["labels"]
    low_label = labels["low_freq"]
    high_label = labels["high_freq"]
    unclassified_label = labels["unclassified"]

    fi_values = valid_df["fi"].dropna().values

    threshold, threshold_method, threshold_debug = find_fi_threshold_from_histogram(
        fi_values=fi_values,
        classification_cfg=classification_cfg,
    )

    if len(valid_df) > 0:
        if threshold is None:
            valid_df["fi_class"] = unclassified_label
        else:
            valid_df["fi_class"] = np.where(valid_df["fi"] < threshold, low_label, high_label)

    return valid_df, threshold, threshold_method, threshold_debug


In [ ]:

# %% [markdown]
# ## 6. Load data

# %%
catalog, catalog_colmap = load_catalog(PATHS["catalog_csv"])
picks = load_picks(PATHS["picks_csv"])
stations = load_stations(PATHS["stations_json"])
inventory = load_inventory(PATHS["inventory_xml"])

print(f"Catalog events: {len(catalog)}")
print(f"Picks: {len(picks)}")
print(f"Stations: {len(stations)}")
print(f"Inventory loaded: {inventory is not None}")

catalog_full, catalog_filtered = filter_catalog_by_radius(
    catalog,
    VOLCANO,
    FILTERS["event_radius_km"],
    lat_col=catalog_colmap["latitude"],
    lon_col=catalog_colmap["longitude"],
)

catalog_filtered[catalog_colmap["event_id"]] = normalize_event_id(catalog_filtered[catalog_colmap["event_id"]])

print(f"Events inside {FILTERS['event_radius_km']} km: {len(catalog_filtered)} / {len(catalog)}")

catalog_ids = set(catalog_filtered[catalog_colmap["event_id"]].astype(str).unique())
pick_ids = set(picks["event_id_link"].astype(str).unique())

print(f"Matching event IDs (catalog vs picks): {len(catalog_ids & pick_ids)}")
print("Sample common IDs:", sorted(list(catalog_ids & pick_ids))[:10])


In [ ]:

# %% [markdown]
# ## 7. Build fast lookup indexes

# %%
chunk_index = build_chunk_index(PATHS["data_base"])
chunk_starts, chunk_ends, chunk_folders = prepare_chunk_lookup(chunk_index)

picks_by_event = build_picks_by_event_lookup(picks)

event_station_distance_lookup = build_event_station_distance_lookup(
    catalog_filtered=catalog_filtered,
    stations=stations,
    catalog_colmap=catalog_colmap,
)

print(f"Chunk folders found: {len(chunk_index)}")
print(f"Events with P-pick lookup: {len(picks_by_event)}")
print(f"Event-station distance maps built: {len(event_station_distance_lookup)}")

# %% [markdown]
# ## 8. Process events and compute FI

# %%
results = []

event_rows = catalog_filtered.to_dict("records")

for i, event_row in enumerate(event_rows, start=1):
    event_result = process_event(
        event_row=event_row,
        picks_by_event=picks_by_event,
        event_station_distance_lookup=event_station_distance_lookup,
        data_base=PATHS["data_base"],
        windows=WINDOWS,
        filters=FILTERS,
        freq_bands=FREQ_BANDS,
        inventory=inventory,
        catalog_colmap=catalog_colmap,
        chunk_starts=chunk_starts,
        chunk_ends=chunk_ends,
        chunk_folders=chunk_folders,
    )
    results.append(event_result)

    if i % 10 == 0 or i == len(event_rows):
        print(f"Processed {i}/{len(event_rows)} events")

results_df = pd.DataFrame(results)

print("\nStatus counts:")
print(results_df["status"].value_counts())

valid_df = results_df[results_df["status"] == "ok"].copy()
print(f"\nEvents with a valid FI: {len(valid_df)}")


In [ ]:

# %% [markdown]
# ## 9. Classify valid events

# %%
valid_df, threshold, threshold_method, threshold_debug = classify_fi_values(
    valid_df=valid_df,
    classification_cfg=CLASSIFICATION,
)

print(f"Threshold = {threshold}")
print(f"Method = {threshold_method}")

print("\nFinal classification in valid_df:")
print(valid_df["fi_class"].value_counts(dropna=False))


In [ ]:

# %% [markdown]
# ## 10. FI histogram

# %%
if len(valid_df) > 0:
    fi_vals = valid_df["fi"].dropna().values

    LP_COLOR = "#3B82F6"
    VT_COLOR = "#F59E0B"
    THR_COLOR = "#B91C1C"
    SMOOTH_COLOR = "#7C3AED"
    MIN_COLOR = "#DC2626"

    fig, ax = plt.subplots(figsize=(10, 6))

    counts, bins, patches = ax.hist(
        fi_vals,
        bins=PLOT["hist_bins"],
        density=False,
        edgecolor="black",
        linewidth=1.0,
        alpha=0.85
    )

    if threshold is not None:
        for left, right, patch in zip(bins[:-1], bins[1:], patches):
            center = 0.5 * (left + right)
            if center < threshold:
                patch.set_facecolor(LP_COLOR)
            else:
                patch.set_facecolor(VT_COLOR)

    counts_smooth = threshold_debug.get("counts_smooth", None)
    bin_centers = threshold_debug.get("bin_centers", None)
    minima = threshold_debug.get("internal_minima", None)
    selected_minimum = threshold_debug.get("selected_minimum", None)

    if bin_centers is not None and counts_smooth is not None:
        ax.plot(
            bin_centers,
            counts_smooth,
            color=SMOOTH_COLOR,
            linewidth=2.2,
            marker="o",
            label="Smoothed bin counts"
        )

    if minima is not None and bin_centers is not None and counts_smooth is not None and len(minima) > 0:
        ax.scatter(
            bin_centers[minima],
            counts_smooth[minima],
            color=MIN_COLOR,
            s=60,
            zorder=5,
            label="Internal minima"
        )

    if selected_minimum is not None and bin_centers is not None and counts_smooth is not None:
        ax.scatter(
            [bin_centers[selected_minimum]],
            [counts_smooth[selected_minimum]],
            color=THR_COLOR,
            edgecolors="black",
            s=110,
            zorder=6,
            label="Selected threshold bin"
        )

    if threshold is not None:
        ax.axvline(
            threshold,
            color=THR_COLOR,
            linestyle="--",
            linewidth=2.2,
            label=f"Threshold = {threshold:.3f}"
        )
        ax.axvspan(bins[0], threshold, color=LP_COLOR, alpha=0.05, zorder=0)
        ax.axvspan(threshold, bins[-1], color=VT_COLOR, alpha=0.05, zorder=0)

    ax.set_xlabel("Frequency Index (FI)")
    ax.set_ylabel("Event count")
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig(PATHS["output_dir"] / "fi_histogram_threshold.png", dpi=150)
    plt.show()
else:
    print("No valid events to plot the histogram.")


In [ ]:

# %% [markdown]
# ## 11. Save the final catalog with FI

## %%
event_id_col = catalog_colmap["event_id"]

catalog_filtered = catalog_filtered.copy()
catalog_filtered[event_id_col] = normalize_event_id(catalog_filtered[event_id_col])

if len(valid_df) > 0:
    fi_class_df = (
        valid_df[["event_id", "fi_class"]]
        .drop_duplicates(subset=["event_id"])
        .copy()
    )
    fi_class_df["event_id"] = normalize_event_id(fi_class_df["event_id"])
else:
    fi_class_df = pd.DataFrame(columns=["event_id", "fi_class"])

output = catalog_filtered.copy().merge(
    fi_class_df,
    left_on=event_id_col,
    right_on="event_id",
    how="left"
)

output["FI"] = output["fi_class"].fillna(CLASSIFICATION["labels"]["unclassified"])

cols_to_drop_from_output = ["r_km", "inside_radius"]

original_cols = [
    c for c in catalog_filtered.columns
    if c not in cols_to_drop_from_output
]

output = output[original_cols + ["FI"]]

output.to_csv(PATHS["output_csv"], index=False)

print(f"Results saved to: {PATHS['output_csv']}")
print(f"Events saved: {len(output)}")

print("\nClass summary in output:")
print(output["FI"].value_counts(dropna=False))

In [ ]:

# %% [markdown]
# ## 12. Figuras FI vs Depth y FI vs Time

# %%
if len(valid_df) > 0:
    LP_COLOR = "#3B82F6"
    VT_COLOR = "#F59E0B"
    UNR_COLOR = "#6B7280"
    THR_COLOR = "#B91C1C"
    GRID_ALPHA = 0.18

    low_label = CLASSIFICATION["labels"]["low_freq"]
    high_label = CLASSIFICATION["labels"]["high_freq"]
    unclassified_label = CLASSIFICATION["labels"]["unclassified"]

    plot_df = valid_df.copy().sort_values("event_time").reset_index(drop=True)
    lp_df = plot_df[plot_df["fi_class"] == low_label].copy()
    vt_df = plot_df[plot_df["fi_class"] == high_label].copy()
    unr_df = plot_df[plot_df["fi_class"] == unclassified_label].copy()

    # FI vs Depth
    fig_depth, ax_depth = plt.subplots(figsize=(8.5, 6))

    if "event_depth_km" in plot_df.columns and plot_df["event_depth_km"].notna().any():
        if len(lp_df) > 0:
            ax_depth.scatter(
                lp_df["fi"], lp_df["event_depth_km"],
                s=54, c=LP_COLOR, edgecolors="black", linewidth=0.45,
                alpha=0.82, label=low_label
            )

        if len(vt_df) > 0:
            ax_depth.scatter(
                vt_df["fi"], vt_df["event_depth_km"],
                s=54, c=VT_COLOR, edgecolors="black", linewidth=0.45,
                alpha=0.82, label=high_label
            )

        if len(unr_df) > 0:
            ax_depth.scatter(
                unr_df["fi"], unr_df["event_depth_km"],
                s=54, c=UNR_COLOR, edgecolors="black", linewidth=0.45,
                alpha=0.82, label=unclassified_label
            )

        if threshold is not None:
            ax_depth.axvline(threshold, color=THR_COLOR, linestyle="--", linewidth=2.0)

        ax_depth.set_xlabel("Frequency Index (FI)")
        ax_depth.set_ylabel("Depth (km)")
        ax_depth.set_title("Frequency Index vs Depth")
        ax_depth.grid(True, alpha=GRID_ALPHA)
        ax_depth.invert_yaxis()
        ax_depth.legend(frameon=True, facecolor="white", edgecolor="0.85")

        plt.tight_layout()
        fig_depth.savefig(PATHS["output_dir"] / "fi_vs_depth.png", bbox_inches="tight", dpi=150)
        fig_depth.savefig(PATHS["output_dir"] / "fi_vs_depth.pdf", bbox_inches="tight")
        plt.show()
    else:
        print("No valid depth column for FI vs Depth.")

    # FI vs Time
    fig_time, ax_time = plt.subplots(figsize=(10, 6))

    if len(lp_df) > 0:
        ax_time.scatter(
            lp_df["event_time"], lp_df["fi"],
            s=42, c=LP_COLOR, edgecolors="black", linewidth=0.4,
            alpha=0.82, label=low_label
        )

    if len(vt_df) > 0:
        ax_time.scatter(
            vt_df["event_time"], vt_df["fi"],
            s=42, c=VT_COLOR, edgecolors="black", linewidth=0.4,
            alpha=0.82, label=high_label
        )

    if len(unr_df) > 0:
        ax_time.scatter(
            unr_df["event_time"], unr_df["fi"],
            s=42, c=UNR_COLOR, edgecolors="black", linewidth=0.4,
            alpha=0.82, label=unclassified_label
        )

    if threshold is not None:
        ax_time.axhline(threshold, color=THR_COLOR, linestyle="--", linewidth=2.0)

    ax_time.set_xlabel("Time")
    ax_time.set_ylabel("Frequency Index (FI)")
    ax_time.set_title("Frequency Index Through Time")
    ax_time.grid(True, alpha=GRID_ALPHA)

    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax_time.xaxis.set_major_locator(locator)
    ax_time.xaxis.set_major_formatter(formatter)

    legend_elements = []
    if len(lp_df) > 0:
        legend_elements.append(
            Line2D([0], [0], marker="o", color="w", markerfacecolor=LP_COLOR,
                   markeredgecolor="black", markersize=8, label=low_label)
        )
    if len(vt_df) > 0:
        legend_elements.append(
            Line2D([0], [0], marker="o", color="w", markerfacecolor=VT_COLOR,
                   markeredgecolor="black", markersize=8, label=high_label)
        )
    if len(unr_df) > 0:
        legend_elements.append(
            Line2D([0], [0], marker="o", color="w", markerfacecolor=UNR_COLOR,
                   markeredgecolor="black", markersize=8, label=unclassified_label)
        )
    if threshold is not None:
        legend_elements.append(
            Line2D([0], [0], color=THR_COLOR, linestyle="--", linewidth=2,
                   label=f"Threshold = {threshold:.2f}")
        )

    if len(legend_elements) > 0:
        ax_time.legend(handles=legend_elements, frameon=True, facecolor="white", edgecolor="0.85")

    plt.tight_layout()
    fig_time.savefig(PATHS["output_dir"] / "fi_vs_time.png", bbox_inches="tight", dpi=150)
    fig_time.savefig(PATHS["output_dir"] / "fi_vs_time.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No valid events to plot.")